# Minimal test: is K=1 negative NLL caused by GPU TF32/cancellation?

In [1]:
from pathlib import Path
import json, sys, torch
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent.resolve()
sys.path.insert(0, str(REPO / "src"))

from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index
from dalg.models.mfa import load_mfa

RUN = REPO / "dalg-cache/pile_wikipedia_gemma2b_mfa_100k/q337_k_sweep/layer17_1_337_mfa"
SHARD = REPO / "dalg-cache/pile_gemma2b_activations"

info = json.loads((RUN / "val_indices.json").read_text())
meta = load_meta_index(SHARD, layer=17)
wanted = set(info["val_global_rows"])
val_pos = [i for i, row in enumerate(meta) if row["global_row"] in wanted]

ds = ActivationBatchDataset(
    SHARD, layer=17, row_subset=val_pos,
    batch_size=2048, drop_prefix=32,
    shuffle_shards=False, shuffle_within_shard=False,
    dtype=torch.float16,
)
X = torch.cat(list(DataLoader(ds, batch_size=None, num_workers=0)), dim=0).float()
model = load_mfa(RUN / "mfa_model.pt", map_location="cpu", dtype=torch.float32).eval()
ckpt = torch.load(RUN / "checkpoint.pt", map_location="cpu", weights_only=False)

print("torch", torch.__version__, "cuda build", torch.version.cuda, "cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("X", X.shape, X.dtype)
print("checkpoint best_metric", ckpt["best_metric"])
print("checkpoint last_val_metric", ckpt["last_val_metric"])

torch 2.11.0+cu128 cuda build 12.8 cuda available True
NVIDIA A100-SXM4-40GB
X torch.Size([10080, 2048]) torch.float32
checkpoint best_metric -1025286.573015873
checkpoint last_val_metric -809858.1498015873


In [2]:
@torch.no_grad()
def eval_nll(device, *, precision=None, tf32=None):
    old_precision = torch.get_float32_matmul_precision()
    old_tf32 = torch.backends.cuda.matmul.allow_tf32
    if precision is not None:
        torch.set_float32_matmul_precision(precision)
    if tf32 is not None:
        torch.backends.cuda.matmul.allow_tf32 = tf32
    try:
        m = model.to(device)
        x = X.to(device)
        vals = []
        for i in range(0, x.shape[0], 2048):
            xb = x[i:i+2048]
            vals.append(float(m.nll(xb).cpu()) * xb.shape[0])
        return sum(vals) / x.shape[0]
    finally:
        model.to("cpu")
        torch.set_float32_matmul_precision(old_precision)
        torch.backends.cuda.matmul.allow_tf32 = old_tf32


print("CPU:", eval_nll("cpu"))
if torch.cuda.is_available():
    print("CUDA high, TF32 on:", eval_nll("cuda", precision="high", tf32=True))
    print("CUDA highest, TF32 off:", eval_nll("cuda", precision="highest", tf32=False))
else:
    print("CUDA unavailable in this Python process")

CPU: 6479.670366753472


CUDA high, TF32 on: -1086085.65
CUDA highest, TF32 off: 6491.253357514881
